# Week 16, Day 5: Decisions and Gaussian classifiers: from posteriors to choices

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w016/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Three losses, three answers

A posterior over a neuron's firing rate is skewed (a gamma shape). Compute its mean, median and mode. Then, for each candidate answer on the grid, compute the expected squared loss, absolute loss and zero-one loss (zero within 0.25 Hz of the truth), and store the best answer for each in the dictionary `best`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
grid = np.linspace(0, 40, 4001)
post = grid ** 2 * np.exp(-grid / 4)                 # a skewed posterior over rate (Hz)
post /= post.sum()

post_mean = ...
post_median = ...                                     # first grid value where the CDF reaches 0.5
post_mode = ...

cands = grid[::10]
exp_sq = [np.sum((a - grid) ** 2 * post) for a in cands]
exp_abs = ...
exp_01 = [1 - post[np.abs(grid - a) < 0.25].sum() for a in cands]
best = {"squared": cands[np.argmin(exp_sq)], "absolute": ..., "zero-one": cands[np.argmin(exp_01)]}
print(f"mean {post_mean:.2f}, median {post_median:.2f}, mode {post_mode:.2f}");  print({k: round(float(v), 2) for k, v in best.items()})

In [ ]:
import numpy as np
def test_point_estimates():
    "mean, median and mode are computed from the posterior"
    assert abs(post_mean - np.sum(grid * post)) < 1e-9 and post_mode == grid[np.argmax(post)]

def test_minimisers():
    "grid search over candidates 0.1 apart confirms each estimate minimises its own expected loss"
    step = 0.15
    assert abs(best["squared"] - post_mean) < step and abs(best["absolute"] - post_median) < step and abs(best["zero-one"] - post_mode) < step

test_point_estimates()
test_minimisers()
print("All checks passed")

### Exercise: Correlations matter for decoding

Write `lda_fit(X, y, diagonal=False)` that estimates the class means and the pooled covariance (keeping only its diagonal if `diagonal=True`) and returns w and b for equal priors. Train on 400 trials, test on 4,000, and compare `acc_full` with `acc_diag`.

In [ ]:
rng = np.random.default_rng(1936)
Sig = np.array([[1.0, 0.8], [0.8, 1.0]])               # positively correlated noise
mu0, mu1 = np.array([0.0, 0.0]), np.array([0.6, -0.2])  # B raises neuron 1 and lowers neuron 2 a little
def make(n):
    y = rng.integers(0, 2, n)
    X = rng.multivariate_normal([0, 0], Sig, n) + np.where(y[:, None] == 1, mu1, mu0)
    return X, y
Xtr, ytr = make(400); Xte, yte = make(4000)

def pooled_cov(X, y):
    R = np.vstack([X[y == c] - X[y == c].mean(0) for c in (0, 1)])
    return R.T @ R / (len(X) - 2)

def lda_fit(X, y, diagonal=False):
    m0, m1 = X[y == 0].mean(0), X[y == 1].mean(0)
    S = pooled_cov(X, y)
    if diagonal:
        S = np.diag(np.diag(S))
    w = ...
    b = ...                                             # boundary halfway between the means: -w . (m0 + m1) / 2
    return w, b

def accuracy(w, b, X, y): return np.mean(((X @ w + b) > 0) == (y == 1))
acc_full = accuracy(*lda_fit(Xtr, ytr), Xte, yte)
acc_diag = accuracy(*lda_fit(Xtr, ytr, diagonal=True), Xte, yte)
print(f"test accuracy: LDA with correlations {acc_full:.3f}, ignoring correlations {acc_diag:.3f}")

In [ ]:
import numpy as np
def test_fit():
    "lda_fit returns w = inverse covariance times the mean difference"
    w, b = lda_fit(Xtr, ytr)
    m0, m1 = Xtr[ytr == 0].mean(0), Xtr[ytr == 1].mean(0)
    assert np.allclose(np.linalg.solve(pooled_cov(Xtr, ytr), m1 - m0), w)

def test_better():
    "on test trials, full LDA is at least 5 points more accurate than the decoder that ignores correlations"
    assert acc_full > acc_diag + 0.05

test_fit()
test_better()
print("All checks passed")

### Exercise: Two gradient estimators

For f(z) = z² with z ~ N(μ = 1.5, σ = 1), the true gradient with respect to μ is 2μ. Compute 5,000 single-sample gradient estimates with the reparameterisation trick (`g_rep` = f′(z) = 2z) and with the score-function estimator (`g_score` = f(z)(z − μ)/σ²), using the same ε, and compare their means and variances.

In [ ]:
mu, sigma = 1.5, 1.0
eps = rng.normal(size=5000)
z = mu + sigma * eps
g_rep = ...
g_score = ...
print(f"true gradient {2 * mu};  reparameterisation mean {g_rep.mean():.3f} (variance {g_rep.var():.2f});  score function mean {g_score.mean():.3f} (variance {g_score.var():.2f})")

In [ ]:
import numpy as np
def test_unbiased():
    "both estimators average close to the true gradient 2 mu (within 0.1)"
    assert abs(np.mean(g_rep) - 2 * mu) < 0.1 and abs(np.mean(g_score) - 2 * mu) < 0.1

def test_variance():
    "the reparameterisation estimator has at least 4 times smaller variance"
    assert np.var(g_score) > 4 * np.var(g_rep)

test_unbiased()
test_variance()
print("All checks passed")